## Setup:

In [1]:
%%capture
!pip install faiss-gpu google-cloud-bigquery google-cloud-storage \
             gcsfs pyarrow pandas torch --quiet

In [2]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 111.7 MB/s eta 0:00:00


In [3]:
!git clone -b dev https://github.com/sbnikhil/RecoSys.git

Cloning into 'RecoSys'...
remote: Enumerating objects: 307, done.
remote: Counting objects: 100% (307/307), done.
remote: Compressing objects: 100% (223/223), done.
remote: Total 307 (delta 146), reused 217 (delta 77), pack-reused 0 (from 0)
Receiving objects: 100% (307/307), 706.57 KiB | 15.03 MiB/s, done.
Resolving deltas: 100% (146/146), done.


In [4]:
from google.colab import files
import os

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No file uploaded")

# Use the real filename from the upload
fname = next(iter(uploaded.keys()))
path = os.path.abspath(fname)
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = path
print("GOOGLE_APPLICATION_CREDENTIALS =", path)

Saving recosys-service-account.json to recosys-service-account.json
GOOGLE_APPLICATION_CREDENTIALS = /content/recosys-service-account.json


In [5]:
import sys
%cd /content/RecoSys
sys.path.insert(0, '/content/RecoSys')
!ls

/content/RecoSys
LICENSE  notebooks  README.md  reports	requirements.txt  scripts  src


In [6]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("WARNING: No GPU. Go to Runtime > Change runtime type > T4 GPU")

Device: cuda
GPU: NVIDIA A100-SXM4-40GB
Memory: 42.4 GB


In [7]:
# Upload the artifacts/500k/ folder contents from your local machine.
# Easiest: zip it locally, upload here, unzip.
# Run this on your LOCAL machine first:
#   cd artifacts && zip -r 500k_artifacts.zip 500k/
# Then upload the zip here:

from google.colab import files
import zipfile

print("Upload 500k.zip:")
uploaded = files.upload()
with zipfile.ZipFile("500k.zip", "r") as z:
    z.extractall("artifacts/")
print("Artifacts extracted:")
!ls artifacts/500k/

Upload 500k.zip:


Saving 500k.zip to 500k.zip
Artifacts extracted:
ls: cannot access 'artifacts/500k/': No such file or directory


In [9]:
import pathlib, json
assert pathlib.Path("artifacts/500k/vocabs.pkl").exists(), "Missing vocabs.pkl"
meta = json.loads(pathlib.Path("artifacts/500k/sequences_v2/metadata.json").read_text())
print(f"train_sessions: {meta['n_train_sessions']:,}")
print(f"val_sessions  : {meta['n_val_sessions']:,}")
print(f"n_items       : {meta['n_items']:,}")
print("Artifacts OK")

train_sessions: 1,450,480
val_sessions  : 76,304
n_items       : 284,523
Artifacts OK


## GRU4Rec Training:

In [10]:
# If you need to rebuild the session sequences on Colab first:
# !python scripts/sequence/build_session_sequences.py

In [12]:
# Then kick off overnight training:
!python scripts/sequence/train_gru4rec_session.py
# A100 default: batch_size=256 (~11 GB peak VRAM)
# T4/V100: add --batch-size 64 or --batch-size 128


  GRU4Rec Session Training V9
  Device          : cuda
  Artifacts dir   : /content/RecoSys/artifacts/500k
  Sequences dir   : /content/RecoSys/artifacts/500k/sequences_v2
  Checkpoint dir  : /content/RecoSys/artifacts/500k/sequences_v2/checkpoints_v9_gru4rec_session
  Batch size      : 256
  Max epochs      : 30
  Patience        : 5
  embed_dim       : 128
  gru_hidden      : 256  (n_layers=1)
  dropout         : 0.3
  lr              : 0.0003  wd=1e-05
  temperature     : 0.07
  scheduler       : cosine  (lr_min=1e-05)
  label_smoothing : 0.1

  > Loading vocabs...
     445,150 users  /  284,524 items (incl. PAD)

  > Loading session parquets...
     train: 1,450,480 sessions
     val  : 76,304 sessions
     max_seq_len: 20

  Building datasets
  Padded train_sessions: 1,450,480 sessions -> (1,450,480 x 20) in 2.4s (~464 MB)
  SessionEvalDataset: 76,304 sessions, prefix shape (76,304 x 20) in 0.2s

  Train loader : 5,666 batches x 256

  Model
GRU4RecModel:
  n_items     : 284,524


In [15]:
# ── Save GRU4Rec V9 artifacts to GCS ─────────────────────────────────────────
import os, json, pathlib
from datetime import datetime, timezone
from google.cloud import storage

# Auth — same as in notebook 07
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/recosys-service-account.json"

BUCKET_NAME = "recosys-data-bucket"
GCS_PREFIX  = "models/gru4rec_session_v9"

REPO_ROOT = pathlib.Path("/content/RecoSys")
CKPT_DIR  = REPO_ROOT / "artifacts/500k/sequences_v2/checkpoints_v9_gru4rec_session"

# Tag the folder with best epoch + NDCG so runs don't collide
log_path = CKPT_DIR / "training_log.json"
if log_path.exists():
    log  = json.loads(log_path.read_text())
    best = max(log, key=lambda r: r["val_ndcg_20"])
    tag  = f"ep{best['epoch']}_ndcg{best['val_ndcg_20']:.4f}"
else:
    tag  = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M")

dest_prefix = f"{GCS_PREFIX}/{tag}"

# Upload every file in the checkpoint dir
client = storage.Client()
to_upload = list(CKPT_DIR.glob("*"))
print(f"Uploading {len(to_upload)} file(s) to gs://{BUCKET_NAME}/{dest_prefix}/\n")

for p in to_upload:
    blob_path = f"{dest_prefix}/{p.name}"
    bucket    = client.bucket(BUCKET_NAME)
    blob      = bucket.blob(blob_path)
    blob.upload_from_filename(str(p))
    print(f"  gs://{BUCKET_NAME}/{blob_path}  <=  {p.name}")

print(f"\nDone. Artifacts at: gs://{BUCKET_NAME}/{dest_prefix}/")


Uploading 4 file(s) to gs://recosys-data-bucket/models/gru4rec_session_v9/ep14_ndcg0.2606/

  gs://recosys-data-bucket/models/gru4rec_session_v9/ep14_ndcg0.2606/best_checkpoint.pt  <=  best_checkpoint.pt
  gs://recosys-data-bucket/models/gru4rec_session_v9/ep14_ndcg0.2606/training_log.json  <=  training_log.json
  gs://recosys-data-bucket/models/gru4rec_session_v9/ep14_ndcg0.2606/latest_checkpoint.pt  <=  latest_checkpoint.pt
  gs://recosys-data-bucket/models/gru4rec_session_v9/ep14_ndcg0.2606/hparams.json  <=  hparams.json

Done. Artifacts at: gs://recosys-data-bucket/models/gru4rec_session_v9/ep14_ndcg0.2606/
